In [ ]:
!pip install -q transformers datasets evaluate sacrebleu accelerate

In [ ]:
import os
import torch
import numpy as np
import evaluate
from datasets import load_from_disk
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

# 1. Duong dan va nap Dataset
DATASET_PATH = "/kaggle/input/notebooks/khnhhngnguynhu/preprocess-phomt-mariannmt/PhoMT/datasets_en2vi"
print("Loading datasets at:", DATASET_PATH)
datasets_en2vi = load_from_disk(DATASET_PATH)
print(datasets_en2vi)

# 2. Nap Model va Tokenizer Pretrained
MODEL_NAME = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# 3. Data Collator va Metric tinh SacreBLEU
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Thay thiet lap token -100 thanh pad_token_id de decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_predictions = [pred.strip() for pred in decoded_predictions]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_predictions, references=decoded_labels)
    return {"bleu": round(result["score"], 2)}

# 4. Cau hinh Hyperparameters
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/checkpoints_marian_en2vi",
    # eval_strategy="steps",
    # eval_steps=2000,
    # save_strategy="steps",
    # save_steps=2000,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,    # Effective batch size = 64
    num_train_epochs=2,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_num_beams=4,           # Kich hoat Beam Search khi danh gia
    fp16=True,                        # Bat Mixed Precision
    logging_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none"
)

# 5. Khoi tao Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets_en2vi["train"],
    eval_dataset=datasets_en2vi["validation"],
    data_collator=data_collator,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 6. Huan luyen Mo hinh
trainer.train()

# 7. Luu Checkpoint Tot Nhat
SAVE_DIRECTORY = "/kaggle/working/en2vi"
trainer.save_model(SAVE_DIRECTORY)
tokenizer.save_pretrained(SAVE_DIRECTORY)
print("Saved at:", SAVE_DIRECTORY)

In [ ]:
# from pathlib import Path
# from datasets import Dataset, DatasetDict

# detokenization_path = Path("/kaggle/input/notebooks/khnhhngnguynhu/preprocess-phomt-mariannmt/vinai/PhoMT/detokenization")
# if not detokenization_path.exists():
#     detokenization_path = Path("/kaggle/input/vinai/PhoMT/detokenization")

# def load_parallel_subset(path, split_name, max_samples=None):
#     en_file = path / split_name / f"{split_name}.en"
#     vi_file = path / split_name / f"{split_name}.vi"
    
#     src_lines = []
#     tgt_lines = []
    
#     with open(en_file, "r", encoding="utf-8") as f_en, open(vi_file, "r", encoding="utf-8") as f_vi:
#         for idx, (en_line, vi_line) in enumerate(zip(f_en, f_vi)):
#             en_str = en_line.strip()
#             vi_str = vi_line.strip()
            
#             # Lọc bỏ câu rỗng hoặc quá ngắn/quá dài
#             if 3 <= len(en_str.split()) <= 80 and 3 <= len(vi_str.split()) <= 100:
#                 src_lines.append(en_str)
#                 tgt_lines.append(vi_str)
                
#             if max_samples and len(src_lines) >= max_samples:
#                 break
                
#     return Dataset.from_dict({"en": src_lines, "vi": tgt_lines})

# # Lấy 150.000 câu train và 2.000 câu validation
# raw_train = load_parallel_subset(detokenization_path, "train", max_samples=150000)
# raw_val = load_parallel_subset(detokenization_path, "dev", max_samples=2000)

# raw_datasets = DatasetDict({
#     "train": raw_train,
#     "validation": raw_val
# })

# print(f"Train samples: {len(raw_datasets['train']):,}")
# print(f"Validation samples: {len(raw_datasets['validation']):,}")

In [ ]:
# from transformers import AutoTokenizer

# model_checkpoint = "Helsinki-NLP/opus-mt-en-vi"
# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# max_length = 128

# def preprocess_function(examples):
#     inputs = examples["en"]
#     targets = examples["vi"]
    
#     model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
#     labels = tokenizer(text_target=targets, max_length=max_length, truncation=True)
    
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

# tokenized_datasets = raw_datasets.map(
#     preprocess_function,
#     batched=True,
#     batch_size=5000,
#     remove_columns=raw_datasets["train"].column_names
# )

In [ ]:
# import numpy as np
# import evaluate
# from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

# model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
# data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# metric = evaluate.load("sacrebleu")

# def compute_metrics(eval_preds):
#     predictions, labels = eval_preds
#     if isinstance(predictions, tuple):
#         predictions = predictions[0]
        
#     # Thay thế -100 để decode được
#     decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
#     labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
#     # Làm sạch chuỗi
#     decoded_predictions = [pred.strip() for pred in decoded_predictions]
#     decoded_labels = [[label.strip()] for label in decoded_labels]
    
#     result = metric.compute(predictions=decoded_predictions, references=decoded_labels)
#     return {"bleu": round(result["score"], 2)}

In [ ]:
# from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# training_args = Seq2SeqTrainingArguments(
#     output_dir="./marian-finetuned-phomt",
#     eval_strategy="steps",
#     eval_steps=1000,                  # Đánh giá sau mỗi 1000 step
#     save_strategy="steps",
#     save_steps=1000,
#     learning_rate=3e-5,               # Learning rate phù hợp cho fine-tune Marian
#     per_device_train_batch_size=32,   # VRAM T4 16GB chứa tốt batch 32 với max_len 128
#     per_device_eval_batch_size=32,
#     gradient_accumulation_steps=2,    # Effective batch = 64
#     weight_decay=0.01,
#     save_total_limit=2,
#     num_train_epochs=2,               # 2 epochs là đủ mô hình thích nghi tốt với PhoMT
#     predict_with_generate=True,       # Bắt buộc bật để tính BLEU
#     generation_max_length=128,
#     fp16=True,                        # Tăng tốc tính toán trên GPU T4
#     logging_steps=100,
#     load_best_model_at_end=True,
#     metric_for_best_model="bleu",
#     greater_is_better=True,
#     report_to="none"
# )

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
#     data_collator=data_collator,
#     # tokenizer=tokenizer,
#     compute_metrics=compute_metrics
# )

# # Bắt đầu quá trình fine-tune
# trainer.train()

# # Lưu lại mô hình và tokenizer đã fine-tune
# trainer.save_model("./kaggle/working/en2vi.pt")
# tokenizer.save_pretrained("./kaggle/working/en2vi.pt")

In [ ]:
# from transformers import pipeline

# translator = pipeline(
#     "translation", 
#     model="./saved_marian_en2vi", 
#     tokenizer=tokenizer, 
#     device=0
# )

# test_sentences = [
#     "Artificial intelligence is transforming education and technology.",
#     "On August 14th, 1947, a woman goes into labor as the clock ticks towards midnight.",
#     "Vietnam National University is one of the top universities in Vietnam."
# ]

# for sentence in test_sentences:
#     output = translator(sentence, max_length=128)
#     print(f"EN: {sentence}")
#     print(f"VI: {output[0]['translation_text']}\n")